
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# LAB - Build a Feature Engineering Pipeline

In this lab, you will build a complete feature engineering pipeline using the **CDC Diabetes Health Indicators** dataset. You will load and clean the data, create a Spark ML pipeline that handles missing values, encodes categorical features, and scales numerical features, then apply it consistently to both training and test sets. Finally, you will prepare the target column and save the pipeline for future reuse.

**Lab Outline**

* **Task 1:** Load Dataset and Data Preparation
  * **1.1.** Load Dataset
  * **1.2.** Data Preparation — Type Casting, Missing Columns, Outlier Removal, Save Silver Table
* **Task 2:** Split Dataset into Training and Testing Sets
* **Task 3:** Create Feature Engineering Pipeline
  * **3.1.** Analyze Data Types and Missing Values
  * **3.2.** Define Pipeline Stages and Build Pipeline
* **Task 4:** Fit the Pipeline
* **Task 5:** Transform Datasets and Prepare the Target Column
* **Task 6:** Save and Load Pipeline

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #F44336; background: #FFEBEE; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #C62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong> — <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976D2; text-decoration: underline;">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

### Classroom Setup
Run the following cell to configure your working environment for this course.

This setup will:
- Initialize the `DA` object (Databricks Academy helper)
- Configure your **default catalog** and **schema**
- Provision any supporting configuration needed for this lab

**NOTE:** The `DA` object is only available in Databricks Academy courses.

In [0]:
%run ../Includes/Classroom-Setup-2.3

**Other Conventions:**

Throughout this lab, we'll refer to the object `DA`. This object, provided by Databricks Academy, contains **variables such as your username, catalog name, schema name, working directory, and dataset locations**. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")

## Task 1: Load Dataset and Data Preparation

In this task, you will load the **CDC Diabetes Health Indicators** dataset and prepare it for machine learning. This health survey dataset contains demographic information and health indicators about respondents, including whether they have been diagnosed with diabetes.

You will:
- Load the raw CSV file into a Spark DataFrame
- Perform initial data preparation: type casting, removing columns with excessive missing values, and filtering outliers
- Save the cleaned dataset as a **Delta silver table** for downstream use

### Task 1.1: Load the Dataset

Load the CDC diabetes dataset from the provided path. Use the following options:
- `.option("nullValue", "null")` — read the string `"null"` as a SQL `null`
- `header="true"` — the CSV file includes a header row
- `inferSchema="true"` — let Spark automatically detect column data types
- `multiLine="true"` — handle multi-line CSV fields correctly

Once loaded, display the DataFrame to inspect its structure and column types.

In [0]:
## Set the path of the dataset
dataset_path = f"{DA.paths.datasets.cdc_diabetes}/cdc-diabetes/diabetes_binary_5050_raw.csv"

## Read the CSV file using Spark
## Set the header, inferSchema, multiLine, and nullValue options
cdc_df = <FILL_IN>

## Display the resulting DataFrame
<FILL_IN>

##### Task 1.1 — Load Dataset — Solution
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyAnsT1a()" style="background:#1976d2; color:white; border:none; padding:6px 14px; border-radius:6px; cursor:pointer; font-size:0.85rem; margin: 8px 0 4px 0; display:inline-block;">
Copy to clipboard
</button>

<pre id="copy-block-t1a" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;"><code>
# Set the path of the dataset
dataset_path = f"{DA.paths.datasets.cdc_diabetes}/cdc-diabetes/diabetes_binary_5050_raw.csv"

# Read the CSV file using Spark
cdc_df = spark.read.option("nullValue", "null").csv(
    dataset_path,
    header="true",
    inferSchema="true",
    multiLine="true",
    escape='"'
)

# Display the resulting DataFrame
display(cdc_df)
</code></pre>

<script>
function copyAnsT1a() {
  const el = document.getElementById("copy-block-t1a");
  if (!el) return;
  const text = el.innerText;

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackT1a(text);
      });
  } else {
    fallbackT1a(text);
  }
}

function fallbackT1a(text) {
  const ta = document.createElement("textarea");
  ta.value = text;
  ta.style.position = "fixed";
  ta.style.left = "-9999px";
  document.body.appendChild(ta);
  ta.select();

  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    alert("Could not copy. Please copy manually.");
  } finally {
    document.body.removeChild(ta);
  }
}
</script>

</details>

### Task 1.2: Data Preparation

With the data loaded, the next step is to prepare it for modeling. You will:
- **Cast data types** — Convert integer and boolean columns to `double` for compatibility with Spark ML
- **Remove columns with too many missing values** — Drop any column where more than 60% of values are missing
- **Remove outliers** — Filter records with invalid or extreme values
- **Save the cleaned data** — Write to a Delta silver table for reuse

**1.2a — Convert Data Types**

Spark ML requires all feature columns to be numeric. Identify any `IntegerType` or `BooleanType` columns in the DataFrame and cast them to `DoubleType`. This ensures numerical consistency across all features.

> Print the schema after casting to verify the changes.

In [0]:
from pyspark.sql.types import IntegerType, BooleanType
from pyspark.sql.functions import col

## Get a list of integer and boolean columns
integer_cols = <FILL_IN>

## Cast each to double for Spark ML compatibility
for column in integer_cols:
    cdc_df = cdc_df.withColumn(column, <FILL_IN>)

## Print the schema to verify the changes
cdc_df.<FILL_IN>

##### Task 1.2a — Type Casting — Solution
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyAnsT1b()" style="background:#1976d2; color:white; border:none; padding:6px 14px; border-radius:6px; cursor:pointer; font-size:0.85rem; margin: 8px 0 4px 0; display:inline-block;">
Copy to clipboard
</button>

<pre id="copy-block-t1b" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;"><code>
from pyspark.sql.types import IntegerType, BooleanType
from pyspark.sql.functions import col

# Get a list of integer and boolean columns
integer_cols = [
    c.name for c in cdc_df.schema.fields
    if isinstance(c.dataType, (IntegerType, BooleanType))
]

# Cast each to double for Spark ML compatibility
for column in integer_cols:
    cdc_df = cdc_df.withColumn(column, col(column).cast("double"))

# Print the schema to verify the changes
cdc_df.printSchema()
</code></pre>

<script>
function copyAnsT1b() {
  const el = document.getElementById("copy-block-t1b");
  if (!el) return;
  const text = el.innerText;

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackT1b(text);
      });
  } else {
    fallbackT1b(text);
  }
}

function fallbackT1b(text) {
  const ta = document.createElement("textarea");
  ta.value = text;
  ta.style.position = "fixed";
  ta.style.left = "-9999px";
  document.body.appendChild(ta);
  ta.select();

  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    alert("Could not copy. Please copy manually.");
  } finally {
    document.body.removeChild(ta);
  }
}
</script>

</details>

**1.2b — Remove Columns with Too Many Missing Values**

Columns with a very high proportion of missing values are unlikely to be useful for modeling and can cause issues during pipeline fitting. Count the missing values in each column, then drop any column where more than **60%** of rows are null.

> Display the missing value counts before dropping, then display the cleaned DataFrame after.

In [0]:
from pyspark.sql.functions import col, when, sum as spark_sum

## Count missing values per column
missing_counts = cdc_df.agg(*[
    <FILL_IN>
    for c in cdc_df.columns
]).first().asDict()

## Display missing value counts as a summary DataFrame
missing_df = spark.createDataFrame(
    <FILL_IN>,
    ["column", "missing_count"]
)
display(missing_df.orderBy("missing_count", ascending=False))

## Set threshold: drop columns with more than 60% missing data
per_thresh = 0.6
N = cdc_df.<FILL_IN>
to_drop_missing = <FILL_IN>

print(f"Dropping columns with >{per_thresh * 100}% missing data: {to_drop_missing}")
cdc_no_missing_df = cdc_df.drop(*to_drop_missing)
display(cdc_no_missing_df)

##### Task 1.2b — Remove Missing-Heavy Columns — Solution
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyAnsT1c()" style="background:#1976d2; color:white; border:none; padding:6px 14px; border-radius:6px; cursor:pointer; font-size:0.85rem; margin: 8px 0 4px 0; display:inline-block;">
Copy to clipboard
</button>

<pre id="copy-block-t1c" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;"><code>
from pyspark.sql.functions import col, when, sum as spark_sum

# Count missing values per column
missing_counts = cdc_df.agg(*[
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in cdc_df.columns
]).first().asDict()

# Display missing value counts as a summary DataFrame
missing_df = spark.createDataFrame(
    [(c, int(v)) for c, v in missing_counts.items()],
    ["column", "missing_count"]
)
display(missing_df.orderBy("missing_count", ascending=False))

# Set threshold: drop columns with more than 60% missing data
per_thresh = 0.6
N = cdc_df.count()

to_drop_missing = [
    c for c, v in missing_counts.items()
    if v / N >= per_thresh
]

print(f"Dropping columns with >{per_thresh * 100}% missing data: {to_drop_missing}")

cdc_no_missing_df = cdc_df.drop(*to_drop_missing)
display(cdc_no_missing_df)
</code></pre>

<script>
function copyAnsT1c() {
  const el = document.getElementById("copy-block-t1c");
  if (!el) return;
  const text = el.innerText;

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackT1c(text);
      });
  } else {
    fallbackT1c(text);
  }
}

function fallbackT1c(text) {
  const ta = document.createElement("textarea");
  ta.value = text;
  ta.style.position = "fixed";
  ta.style.left = "-9999px";
  document.body.appendChild(ta);
  ta.select();

  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    alert("Could not copy. Please copy manually.");
  } finally {
    document.body.removeChild(ta);
  }
}
</script>

</details>

**1.2c — Remove Outliers**

Before modeling, we remove records with values that fall outside plausible ranges. For this dataset:
- `MentHlth` — number of days of poor mental health; negative values are invalid (cutoff: `>= 0`)
- `BMI` — body mass index; values above 50 are considered extreme outliers (cutoff: `<= 50`)

Apply both filters in a single step, then print the row count before and after.

In [0]:
## Define cutoff values
MentHlth_cutoff = 0   ## MentHlth cannot be negative
BMI_cutoff = 50       ## Reasonable upper limit for BMI

## Apply both filters in a single step
cdc_no_outliers_df = cdc_no_missing_df.filter(
    <FILL_IN>
)

## Print count before and after
print(<FILL_IN>)

## Display the filtered DataFrame
<FILL_IN>

##### Task 1.2c — Remove Outliers — Solution

<details>
<summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyAnsT1c()" style="background:#1976d2; color:white; border:none; padding:6px 14px; border-radius:6px; cursor:pointer; font-size:0.85rem; margin: 8px 0 4px 0; display:inline-block;">
Copy to clipboard
</button>

<pre id="copy-block-t1c" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
## Define cutoff values
MentHlth_cutoff = 0   ## MentHlth cannot be negative
BMI_cutoff = 50       ## Reasonable upper limit for BMI

## Apply both filters in a single step
cdc_no_outliers_df = cdc_no_missing_df.filter(
    (col("MentHlth") >= MentHlth_cutoff) & (col("BMI") <= BMI_cutoff)
)

## Print count before and after
print(f"Row count — Before: {cdc_no_missing_df.count()} / After: {cdc_no_outliers_df.count()}")

## Display the filtered DataFrame
display(cdc_no_outliers_df)
</code>
</pre>

<script>
function copyAnsT1c() {
  const el = document.getElementById("copy-block-t1c");
  if (!el) return;
  const text = el.innerText;

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => fallbackT1c(text));
  } else {
    fallbackT1c(text);
  }
}

function fallbackT1c(text) {
  const ta = document.createElement("textarea");
  ta.value = text;
  ta.style.position = "fixed";
  ta.style.left = "-9999px";
  document.body.appendChild(ta);
  ta.select();

  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    alert("Could not copy. Please copy manually.");
  } finally {
    document.body.removeChild(ta);
  }
}
</script>

</details>

**1.2d — Save the Cleaned Dataset as a Silver Table**

Save the cleaned DataFrame as a Delta table. This creates a **silver table** — a cleaned, structured version of the raw data — that can be reused in downstream tasks without re-running the data preparation steps.

In [0]:
cdc_df_full = "cdc_df_full"

## Build the silver table name
cdc_df_full_silver = <FILL_IN>

## Save as Delta table (overwrite if exists)
cdc_no_outliers_df.write.mode("overwrite").option("mergeSchema", True).<FILL_IN>

print(f"Saved silver table: {cdc_df_full_silver}")
display(cdc_no_outliers_df)

##### Task 1.2d — Save Silver Table — Solution
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyAnsT1e()" style="background:#1976d2; color:white; border:none; padding:6px 14px; border-radius:6px; cursor:pointer; font-size:0.85rem; margin: 8px 0 4px 0; display:inline-block;">
Copy to clipboard
</button>

<pre id="copy-block-t1e" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;"><code>
# Base name for the dataset
cdc_df_full = "cdc_df_full"

# Build the silver table name
cdc_df_full_silver = f"{cdc_df_full}_silver"

# Save as Delta table (overwrite if exists)
cdc_no_outliers_df.write.mode("overwrite").option("mergeSchema", True).saveAsTable(cdc_df_full_silver)

print(f"Saved silver table: {cdc_df_full_silver}")

# Display the resulting DataFrame
display(cdc_no_outliers_df)
</code></pre>

<script>
function copyAnsT1e() {
  const el = document.getElementById("copy-block-t1e");
  if (!el) return;
  const text = el.innerText;

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackT1e(text);
      });
  } else {
    fallbackT1e(text);
  }
}

function fallbackT1e(text) {
  const ta = document.createElement("textarea");
  ta.value = text;
  ta.style.position = "fixed";
  ta.style.left = "-9999px";
  document.body.appendChild(ta);
  ta.select();

  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    alert("Could not copy. Please copy manually.");
  } finally {
    document.body.removeChild(ta);
  }
}
</script>

</details>

## Task 2: Split Dataset into Training and Testing Sets

Split the cleaned dataset into training and testing sets using an **80/20 split**. This ensures that:
- The pipeline is **fitted only on training data** (preventing data leakage)
- The test set remains unseen during pipeline fitting, for an unbiased evaluation

After splitting, save both sets as Delta tables for reproducibility.

In [0]:
## Split the dataset: 80% training, 20% testing
train_df, test_df = cdc_no_outliers_df.<FILL_IN>

## Save each split as a Delta table
train_df.write.mode("overwrite").option("overwriteSchema", True).<FILL_IN>
test_df.write.mode("overwrite").option("overwriteSchema", True).<FILL_IN>

print(f"Train rows: {train_df.count()}, Test rows: {test_df.count()}")

##### Task 2 — Split Dataset — Solution
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyAnsT2()" style="background:#1976d2; color:white; border:none; padding:6px 14px; border-radius:6px; cursor:pointer; font-size:0.85rem; margin: 8px 0 4px 0; display:inline-block;">
Copy to clipboard
</button>

<pre id="copy-block-t2" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;"><code>
# Split the dataset: 80% training, 20% testing
train_df, test_df = cdc_no_outliers_df.randomSplit([0.8, 0.2], seed=42)

# Save each split as a Delta table
train_df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(
    f"{DA.catalog_name}.{DA.schema_name}.cdc_df_train"
)

test_df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(
    f"{DA.catalog_name}.{DA.schema_name}.cdc_df_baseline"
)

# Print row counts
print(f"Train rows: {train_df.count()}, Test rows: {test_df.count()}")
</code></pre>

<script>
function copyAnsT2() {
  const el = document.getElementById("copy-block-t2");
  if (!el) return;
  const text = el.innerText;

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackT2(text);
      });
  } else {
    fallbackT2(text);
  }
}

function fallbackT2(text) {
  const ta = document.createElement("textarea");
  ta.value = text;
  ta.style.position = "fixed";
  ta.style.left = "-9999px";
  document.body.appendChild(ta);
  ta.select();

  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    alert("Could not copy. Please copy manually.");
  } finally {
    document.body.removeChild(ta);
  }
}
</script>

</details>

## Task 3: Create Feature Engineering Pipeline

Now you will build a **Spark ML Pipeline** that automates the full feature transformation process. A pipeline ensures that every transformation learned from training data is applied identically to new data — this is the foundation of reproducible, production-ready ML workflows.

**The pipeline will include the following stages:**

| Step | Transformer | Purpose |
|------|------------|---------|
| 1 | `StringIndexer` | Convert string categories to numeric indices |
| 2 | `Imputer` | Fill missing numerical values using the mean |
| 3 | `VectorAssembler` | Combine numerical columns into a single vector |
| 4 | `RobustScaler` | Normalize numerical features, robust to outliers |
| 5 | `OneHotEncoder` | Convert categorical indices to binary sparse vectors |
| 6 | `VectorAssembler` | Combine all features into a final feature vector |

### Task 3.1: Analyze Data Types and Missing Values

Before building the pipeline, analyze the training set to understand which columns need which treatment:

- Cast any remaining `IntegerType` or `BooleanType` columns to `double` (apply to both `train_df` and `test_df`)
- Identify **categorical (string) columns** — these will go through `StringIndexer` → `OneHotEncoder`
- Identify **numerical (double) columns** — these will be imputed and scaled
- Identify **which numeric columns have missing values** — only those need imputation

> **Important:** Exclude the target column `Diabetes_binary` from the feature columns. It is our prediction target, not an input feature.

In [0]:
from pyspark.sql.types import IntegerType, BooleanType, StringType, DoubleType
from pyspark.sql.functions import col, count, when

## Cast any remaining integer and boolean columns to double
## Apply to BOTH train_df and test_df
integer_cols = [c.name for c in train_df.schema.fields if <FILL_IN>]
for column in integer_cols:
    train_df = train_df.withColumn(column, col(column).cast(<FILL_IN>))
    test_df = test_df.withColumn(column, col(column).cast(<FILL_IN>))

## Define the target column — exclude it from all feature lists
target_col = "Diabetes_binary"

## Identify string (categorical) columns — excluding target
string_cols = [c.name for c in train_df.schema.fields if <FILL_IN>]

## Identify numeric columns — excluding target
num_cols = [c.name for c in train_df.schema.fields if <FILL_IN>]

## Find numeric columns with missing values (only these need Imputer)
num_missing_values_logic = [count(when(col(c).isNull(), c)).alias(c) for c in num_cols]
row_dict_num = train_df.select(num_missing_values_logic).first().<FILL_IN>
num_missing_cols = [c for c in row_dict_num if row_dict_num[c] > 0]

print(f"Categorical (string) columns: {string_cols}")
print(f"Numeric columns: {num_cols}")
print(f"Numeric columns with missing values: {num_missing_cols}")

##### Task 3.1 — Analyze Data Types and Missing Values — Solution
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyAnsT3a()" style="background:#1976d2; color:white; border:none; padding:6px 14px; border-radius:6px; cursor:pointer; font-size:0.85rem; margin: 8px 0 4px 0; display:inline-block;">
Copy to clipboard
</button>

<pre id="copy-block-t3a" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;"><code>
from pyspark.sql.types import IntegerType, BooleanType, StringType, DoubleType
from pyspark.sql.functions import col, count, when

# Cast any remaining integer and boolean columns to double
integer_cols = [
    c.name for c in train_df.schema.fields
    if isinstance(c.dataType, (IntegerType, BooleanType))
]

for column in integer_cols:
    train_df = train_df.withColumn(column, col(column).cast("double"))
    test_df = test_df.withColumn(column, col(column).cast("double"))

# Define the target column — exclude it from all feature lists
target_col = "Diabetes_binary"

# Identify string (categorical) columns — excluding target
string_cols = [
    c.name for c in train_df.schema.fields
    if isinstance(c.dataType, StringType) and c.name != target_col
]

# Identify numeric columns — excluding target
num_cols = [
    c.name for c in train_df.schema.fields
    if isinstance(c.dataType, DoubleType) and c.name != target_col
]

# Find numeric columns with missing values
num_missing_values_logic = [
    count(when(col(c).isNull(), c)).alias(c)
    for c in num_cols
]

row_dict_num = train_df.select(num_missing_values_logic).first().asDict()

num_missing_cols = [
    c for c in row_dict_num
    if row_dict_num[c] > 0
]

print(f"Categorical (string) columns: {string_cols}")
print(f"Numeric columns: {num_cols}")
print(f"Numeric columns with missing values: {num_missing_cols}")
</code></pre>

<script>
function copyAnsT3a() {
  const el = document.getElementById("copy-block-t3a");
  if (!el) return;
  const text = el.innerText;

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackT3a(text);
      });
  } else {
    fallbackT3a(text);
  }
}

function fallbackT3a(text) {
  const ta = document.createElement("textarea");
  ta.value = text;
  ta.style.position = "fixed";
  ta.style.left = "-9999px";
  document.body.appendChild(ta);
  ta.select();

  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    alert("Could not copy. Please copy manually.");
  } finally {
    document.body.removeChild(ta);
  }
}
</script>

</details>

### Task 3.2: Define Pipeline Stages and Build the Pipeline

Using the column lists you identified in Task 3.1, define each pipeline stage and assemble them into a `Pipeline`.

Follow these steps in order:
1. **`StringIndexer`** — convert string category columns to numeric indices. Use `handleInvalid="keep"` so null values are treated as a separate category rather than causing an error.
2. **`Imputer`** — fill missing values in numeric columns using the `mean` strategy. Only include columns that actually have missing values (`num_missing_cols`).
3. **`VectorAssembler`** — assemble all numeric feature columns into a single vector called `numerical_assembled`.
4. **`RobustScaler`** — scale the assembled numeric vector. `RobustScaler` uses the interquartile range (IQR) and is less sensitive to remaining outliers than `StandardScaler`.
5. **`OneHotEncoder`** — convert the indexed categorical columns into binary sparse vectors. Use `handleInvalid="keep"`.
6. **`VectorAssembler`** (final) — combine the scaled numeric features and the one-hot encoded categorical features into a single `features` column.
7. **`Pipeline`** — pass all stages to a `Pipeline` in the correct order.

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, Imputer, VectorAssembler, RobustScaler

## Step 1: StringIndexer — string categories to numeric indices
categorical_cols_indexed = [c + "_index" for c in string_cols]
string_indexer = StringIndexer(inputCols=<FILL_IN>, outputCols=<FILL_IN>, handleInvalid=<FILL_IN>)

## Step 2: Imputer — fill missing numeric values with mean
## Only impute columns that have missing values (num_missing_cols)
imputer = Imputer(inputCols=<FILL_IN>, outputCols=<FILL_IN>, strategy=<FILL_IN>)

## Step 3: VectorAssembler — combine numeric columns into one vector
numerical_assembler = VectorAssembler(inputCols=<FILL_IN>, outputCol=<FILL_IN>)

## Step 4: RobustScaler — normalize the numeric vector
numerical_scaler = RobustScaler(inputCol=<FILL_IN>, outputCol=<FILL_IN>)

## Step 5: OneHotEncoder — convert indices to binary sparse vectors
ohe_cols = [c + "_ohe" for c in string_cols]
one_hot_encoder = OneHotEncoder(inputCols=<FILL_IN>, outputCols=<FILL_IN>, handleInvalid=<FILL_IN>)

## Step 6: Final VectorAssembler — combine all features into one vector
feature_cols = ["numerical_scaled"] + <FILL_IN>
vector_assembler = VectorAssembler(inputCols=<FILL_IN>, outputCol=<FILL_IN>)

## Step 7: Build the Pipeline with all stages in order
stages_list = [string_indexer, imputer, numerical_assembler, numerical_scaler, one_hot_encoder, vector_assembler]
pipeline = <FILL_IN>

##### Task 3.2 — Define Pipeline Stages — Solution
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyAnsT3b()" style="background:#1976d2; color:white; border:none; padding:6px 14px; border-radius:6px; cursor:pointer; font-size:0.85rem; margin: 8px 0 4px 0; display:inline-block;">
Copy to clipboard
</button>

<pre id="copy-block-t3b" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;"><code>
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, Imputer, VectorAssembler, RobustScaler

# Define the target column
target_col = "Diabetes_binary"

# ---- Build stages list conditionally ----
stages_list = []

# Step 1: StringIndexer — only if there are string columns
categorical_cols_indexed = [c + "_index" for c in string_cols]
if string_cols:
    string_indexer = StringIndexer(
        inputCols=string_cols,
        outputCols=categorical_cols_indexed,
        handleInvalid="keep"
    )
    stages_list.append(string_indexer)

# Step 2: Imputer — only if there are numeric columns with missing values
if num_missing_cols:
    imputer = Imputer(
        inputCols=num_missing_cols,
        outputCols=num_missing_cols,
        strategy="mean"
    )
    stages_list.append(imputer)

# Step 3: VectorAssembler — combine numeric columns into one vector
numerical_assembler = VectorAssembler(
    inputCols=num_cols,
    outputCol="numerical_assembled"
)
stages_list.append(numerical_assembler)

# Step 4: RobustScaler — normalize the numeric vector
numerical_scaler = RobustScaler(
    inputCol="numerical_assembled",
    outputCol="numerical_scaled"
)
stages_list.append(numerical_scaler)

# Step 5: OneHotEncoder — only if there are categorical columns
ohe_cols = [c + "_ohe" for c in string_cols]
if string_cols:
    one_hot_encoder = OneHotEncoder(
        inputCols=categorical_cols_indexed,
        outputCols=ohe_cols,
        handleInvalid="keep"
    )
    stages_list.append(one_hot_encoder)

# Step 6: Final VectorAssembler — combine all available features
feature_cols = ["numerical_scaled"] + ohe_cols  # ohe_cols will be [] if no string cols
vector_assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
stages_list.append(vector_assembler)

# Step 7: Build the Pipeline
pipeline = Pipeline(stages=stages_list)
</code></pre>

<script>
function copyAnsT3b() {
  const el = document.getElementById("copy-block-t3b");
  if (!el) return;
  const text = el.innerText;

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackT3b(text);
      });
  } else {
    fallbackT3b(text);
  }
}

function fallbackT3b(text) {
  const ta = document.createElement("textarea");
  ta.value = text;
  ta.style.position = "fixed";
  ta.style.left = "-9999px";
  document.body.appendChild(ta);
  ta.select();

  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    alert("Could not copy. Please copy manually.");
  } finally {
    document.body.removeChild(ta);
  }
}
</script>

</details>

## Task 4: Fit the Pipeline

Fit the pipeline on the **training dataset**. During fitting, each Estimator stage (e.g., `StringIndexer`, `Imputer`, `RobustScaler`) learns parameters from `train_df` — such as category mappings, mean values for imputation, and scaling factors. These learned parameters are stored in the resulting `PipelineModel`.

> **Why fit on training data only?** Statistics computed from the test set (e.g., test means used for imputation) would introduce data leakage, making your evaluation results unreliable.

In [0]:
# Fit the pipeline on the training dataset
pipeline_model = <FILL_IN>

##### Task 4 — Fit the Pipeline — Solution
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyAnsT4()" style="background:#1976d2; color:white; border:none; padding:6px 14px; border-radius:6px; cursor:pointer; font-size:0.85rem; margin: 8px 0 4px 0; display:inline-block;">
Copy to clipboard
</button>

<pre id="copy-block-t4" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;"><code>
# Fit the pipeline on the training dataset
pipeline_model = pipeline.fit(train_df)
</code></pre>

<script>
function copyAnsT4() {
  const el = document.getElementById("copy-block-t4");
  if (!el) return;
  const text = el.innerText;

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackT4(text);
      });
  } else {
    fallbackT4(text);
  }
}

function fallbackT4(text) {
  const ta = document.createElement("textarea");
  ta.value = text;
  ta.style.position = "fixed";
  ta.style.left = "-9999px";
  document.body.appendChild(ta);
  ta.select();

  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    alert("Could not copy. Please copy manually.");
  } finally {
    document.body.removeChild(ta);
  }
}
</script>

</details>

## Task 5: Transform Datasets and Prepare the Target Column

Apply the fitted pipeline to both datasets, then prepare the target column for modeling.

**5.1 — Transform datasets:** Use `pipeline_model.transform()` to apply all learned transformations to `train_df` and `test_df`. The result will include the final `features` vector column.

**5.2 — Prepare the target column:** Spark ML models require the target label to be a numeric column named `label`. The `Diabetes_binary` column already contains `0.0` (no diabetes) and `1.0` (diabetes). Create a `label` column from it in both the training and test transformed DataFrames.

> Display the final result showing both `features` and `label` to confirm the dataset is ready for modeling.

In [0]:
## Transform both datasets using the fitted pipeline
train_transformed_df = pipeline_model.<FILL_IN>
test_transformed_df = <FILL_IN>

## Create the 'label' column from 'Diabetes_binary' (already 0.0 / 1.0)
from pyspark.sql.functions import col
train_prepared_df = train_transformed_df.withColumn(<FILL_IN>)
test_prepared_df = test_transformed_df.withColumn(<FILL_IN>)

## Display features and label from the training set
display(train_prepared_df.select(<FILL_IN>))

##### Task 5 — Transform and Prepare Target — Solution
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyAnsT5()" style="background:#1976d2; color:white; border:none; padding:6px 14px; border-radius:6px; cursor:pointer; font-size:0.85rem; margin: 8px 0 4px 0; display:inline-block;">
Copy to clipboard
</button>

<pre id="copy-block-t5" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;"><code>
# Transform both datasets using the fitted pipeline
train_transformed_df = pipeline_model.transform(train_df)
test_transformed_df = pipeline_model.transform(test_df)

# Create the 'label' column from 'Diabetes_binary' (already 0.0 / 1.0)
from pyspark.sql.functions import col

train_prepared_df = train_transformed_df.withColumn("label", col("Diabetes_binary"))
test_prepared_df = test_transformed_df.withColumn("label", col("Diabetes_binary"))

# Display features and label from the training set
display(train_prepared_df.select("features", "label"))
</code></pre>

<script>
function copyAnsT5() {
  const el = document.getElementById("copy-block-t5");
  if (!el) return;
  const text = el.innerText;

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackT5(text);
      });
  } else {
    fallbackT5(text);
  }
}

function fallbackT5(text) {
  const ta = document.createElement("textarea");
  ta.value = text;
  ta.style.position = "fixed";
  ta.style.left = "-9999px";
  document.body.appendChild(ta);
  ta.select();

  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    alert("Could not copy. Please copy manually.");
  } finally {
    document.body.removeChild(ta);
  }
}
</script>

</details>

## Task 6: Save and Load the Pipeline

Saving the fitted pipeline model allows you to reuse the exact same transformations in future sessions or production deployments — without re-fitting from scratch. In this task you will save the pipeline to the working directory, then load it back and inspect its stages.

**6a — Save the Pipeline**

In [0]:
## Save the fitted pipeline model to the working directory
pipeline_model.<FILL_IN>

##### Task 6a — Save Pipeline — Solution
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>
<button onclick="copyAnsT6a()" style="background:#1976d2; color:white; border:none; padding:6px 14px; border-radius:6px; cursor:pointer; font-size:0.85rem; margin: 8px 0 4px 0; display:inline-block;">Copy to clipboard</button>
<pre id="copy-block-t6a" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;">
<code>
## Save the fitted pipeline model to the working directory
pipeline_model.write().overwrite().save(f"{DA.paths.working_dir}/spark_pipelines")
print(f"Pipeline saved to: {DA.paths.working_dir}/spark_pipelines")
</code></pre>
<script>
function copyAnsT6a() {
  const el = document.getElementById("copy-block-t6a");
  if (!el) return;
  const text = el.innerText;
  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => { console.error("Clipboard write failed:", err); fallbackT6a(text); });
  } else { fallbackT6a(text); }
}
function fallbackT6a(text) {
  const ta = document.createElement("textarea");
  ta.value = text; ta.style.position = "fixed"; ta.style.left = "-9999px";
  document.body.appendChild(ta); ta.select();
  try { document.execCommand("copy"); alert("Copied to clipboard"); }
  catch (err) { alert("Could not copy. Please copy manually."); }
  finally { document.body.removeChild(ta); }
}
</script>
</details>

**6b — Load the Saved Pipeline and Inspect Its Stages**

In [0]:
from pyspark.ml import PipelineModel

## Load the saved pipeline model
loaded_pipeline = <FILL_IN>

## Display the pipeline stages to confirm the transformation order
<FILL_IN>

##### Task 6b — Load Pipeline — Solution
<details>
<summary>EXPAND FOR SOLUTION CODE</summary>

<button onclick="copyAnsT6b()" style="background:#1976d2; color:white; border:none; padding:6px 14px; border-radius:6px; cursor:pointer; font-size:0.85rem; margin: 8px 0 4px 0; display:inline-block;">
Copy to clipboard
</button>

<pre id="copy-block-t6b" style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace; border:1px solid #e5e7eb; border-radius:10px; background:#f8fafc; padding:14px 16px; font-size:0.85rem; line-height:1.35; white-space:pre;"><code>
from pyspark.ml import PipelineModel

# Load the saved pipeline model
loaded_pipeline = PipelineModel.load(f"{DA.paths.working_dir}/spark_pipelines")

# Display the pipeline stages to confirm the transformation order
loaded_pipeline.stages
</code></pre>

<script>
function copyAnsT6b() {
  const el = document.getElementById("copy-block-t6b");
  if (!el) return;
  const text = el.innerText;

  if (navigator.clipboard && navigator.clipboard.writeText) {
    navigator.clipboard.writeText(text)
      .then(() => alert("Copied to clipboard"))
      .catch(err => {
        console.error("Clipboard write failed:", err);
        fallbackT6b(text);
      });
  } else {
    fallbackT6b(text);
  }
}

function fallbackT6b(text) {
  const ta = document.createElement("textarea");
  ta.value = text;
  ta.style.position = "fixed";
  ta.style.left = "-9999px";
  document.body.appendChild(ta);
  ta.select();

  try {
    document.execCommand("copy");
    alert("Copied to clipboard");
  } catch (err) {
    alert("Could not copy. Please copy manually.");
  } finally {
    document.body.removeChild(ta);
  }
}
</script>

</details>

## Conclusion

In this lab, you built an end-to-end feature engineering pipeline for the CDC Diabetes Health Indicators dataset using Spark ML.

You performed key data preparation steps, including:
- Casting columns to appropriate data types  
- Removing columns with excessive missing values  
- Filtering out invalid or outlier values  

You then split the dataset into training and test sets and constructed a reusable Spark ML pipeline that:
- Encodes categorical features using `StringIndexer` and `OneHotEncoder`  
- Imputes missing numerical values using `Imputer`  
- Scales numerical features using `RobustScaler`  
- Assembles all features into a single vector for modeling  

You also prepared the target variable by converting `Diabetes_binary` into a numeric `label` column compatible with Spark ML models. Finally, you saved the pipeline for reuse, enabling consistent and reproducible data transformations.

These core practices—structured data preparation, leakage-free transformations, and reusable pipelines—form the foundation for building reliable machine learning workflows in Databricks.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>